# 🐟 Aquaculture Disease Spread Prediction System

## Architecture for 80-90% Explanation of Disease Spread

**Goal:** Build an operational system that:
1. Predicts where next disease outbreak will occur
2. Explains what caused existing outbreaks
3. Reduces future infections through intervention

**Data Sources:**
- 🌊 **Ocean Currents**: NorKyst-800 (800m resolution, daily updates)
- 🚢 **Vessel Movements**: AIS data via BarentsWatch
- 🏭 **Facilities**: 2689 aquaculture sites with disease status
- 🌡️ **Environmental**: Temperature, biomass, production phase

In [ ]:
# Setup
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from scipy.spatial.distance import cdist
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

print("✅ Imports complete")

## 1. Smart Data Filtering (Critical for Performance)

**Key Insight:** Don't process everything all the time!

❌ **Bad:** Track 5000 vessels 24/7, calculate risk for 2689 facilities hourly

✅ **Good:** 
- Only track vessels that visited facilities in last 14 days
- Only calculate risk for facilities within 30km of infected sites
- Run full simulation only on: new outbreak, daily batch (06:00), or manual trigger

In [ ]:
# Mock facility data structure
facilities = pd.DataFrame({
    'id': ['F001', 'F002', 'F003', 'F004', 'F005', 'F006'],
    'name': ['Hammerfest Nord', 'Nordkapp', 'Alta Vest', 'Bergsfjord', 'Hasvik', 'Sørvær'],
    'lat': [70.66, 71.12, 69.97, 70.01, 70.50, 70.45],
    'lon': [23.68, 25.78, 23.27, 22.98, 22.65, 21.95],
    'infected': [True, False, False, False, False, False],
    'disease': ['ILA', None, None, None, None, None],
    'biomass_tons': [850, 920, 640, 780, 590, 810],
    'water_temp_c': [4.2, 4.5, 4.8, 4.3, 4.6, 4.4]
})

def filter_at_risk_facilities(facilities, max_distance_km=30):
    """Filter facilities that are within range of infected sites"""
    infected = facilities[facilities['infected']]
    
    if len(infected) == 0:
        return pd.DataFrame()  # No risk if no infected facilities
    
    # Calculate distances (simplified Haversine)
    def haversine(lat1, lon1, lat2, lon2):
        R = 6371  # Earth radius in km
        dlat = np.radians(lat2 - lat1)
        dlon = np.radians(lon2 - lon1)
        a = np.sin(dlat/2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon/2)**2
        return 2 * R * np.arcsin(np.sqrt(a))
    
    at_risk = []
    for _, facility in facilities[~facilities['infected']].iterrows():
        for _, inf in infected.iterrows():
            dist = haversine(facility['lat'], facility['lon'], inf['lat'], inf['lon'])
            if dist <= max_distance_km:
                at_risk.append(facility['id'])
                break
    
    return facilities[facilities['id'].isin(at_risk)]

at_risk_facilities = filter_at_risk_facilities(facilities, max_distance_km=150)
print(f"📊 Total facilities: {len(facilities)}")
print(f"🔴 Infected: {len(facilities[facilities['infected']])}")
print(f"⚠️  At risk (within 150km): {len(at_risk_facilities)}")
print(f"✅ Performance: Only analyze {len(at_risk_facilities)} / {len(facilities)} facilities (saved {100*(1-len(at_risk_facilities)/len(facilities)):.0f}%)")

## 2. Ocean Current Risk (50-70% of spread)

**Method:** Simplified particle tracking
- Release virtual "virus particles" from infected facilities
- Track movement via ocean currents for 3-14 days (virus survival time)
- Calculate accumulated exposure at each facility

**Data:** NorKyst-800 provides u (eastward) and v (northward) velocity components

In [ ]:
def calculate_current_exposure(source_facility, target_facility, current_velocity_ms=0.15, current_direction_deg=90):
    """
    Calculate exposure from ocean currents using simplified particle tracking
    
    Args:
        source_facility: Infected facility (dict with lat, lon)
        target_facility: Facility to check risk for
        current_velocity_ms: Ocean current velocity in m/s (from NorKyst-800)
        current_direction_deg: Direction currents flow (0=N, 90=E, 180=S, 270=W)
    
    Returns:
        Exposure score (0-100)
    """
    # Calculate distance and bearing from source to target
    def haversine_distance(lat1, lon1, lat2, lon2):
        R = 6371  # km
        dlat = np.radians(lat2 - lat1)
        dlon = np.radians(lon2 - lon1)
        a = np.sin(dlat/2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon/2)**2
        return 2 * R * np.arcsin(np.sqrt(a))
    
    def bearing_degrees(lat1, lon1, lat2, lon2):
        dlat = np.radians(lat2 - lat1)
        dlon = np.radians(lon2 - lon1)
        x = np.sin(dlon) * np.cos(np.radians(lat2))
        y = np.cos(np.radians(lat1)) * np.sin(np.radians(lat2)) - np.sin(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.cos(dlon)
        return (np.degrees(np.arctan2(x, y)) + 360) % 360
    
    distance_km = haversine_distance(
        source_facility['lat'], source_facility['lon'],
        target_facility['lat'], target_facility['lon']
    )
    
    bearing = bearing_degrees(
        source_facility['lat'], source_facility['lon'],
        target_facility['lat'], target_facility['lon']
    )
    
    # Check if current flows toward target (within 45° of direct bearing)
    angle_diff = abs(current_direction_deg - bearing)
    if angle_diff > 180:
        angle_diff = 360 - angle_diff
    
    alignment = max(0, 1 - (angle_diff / 45))  # 1.0 = perfect alignment, 0 = perpendicular
    
    if alignment < 0.5:  # Current not flowing toward target
        return 0
    
    # Calculate virus travel distance over survival period (5 days average)
    virus_survival_days = 5
    virus_travel_km = (current_velocity_ms * 86400 * virus_survival_days) / 1000
    
    # Exposure decreases with distance and dilution
    if distance_km > virus_travel_km:
        return 0  # Virus won't reach
    
    # Calculate exposure score (0-100)
    distance_factor = 1 - (distance_km / virus_travel_km)
    velocity_factor = min(current_velocity_ms / 0.5, 1.0)  # 0.5 m/s = high risk
    
    exposure = 100 * distance_factor * alignment * velocity_factor
    
    return round(exposure, 2)

# Test with example
infected_facility = facilities[facilities['infected']].iloc[0]
target = facilities[~facilities['infected']].iloc[0]

exposure = calculate_current_exposure(
    infected_facility, 
    target,
    current_velocity_ms=0.25,  # From NorKyst-800
    current_direction_deg=85    # Flowing eastward
)

print(f"🌊 Current Exposure Analysis:")
print(f"   Source: {infected_facility['name']} (INFECTED with {infected_facility['disease']})")
print(f"   Target: {target['name']}")
print(f"   Exposure Score: {exposure}/100")
print(f"   Risk Level: {'🔴 HIGH' if exposure > 60 else '🟡 MODERATE' if exposure > 30 else '🟢 LOW'}")

## 3. Vessel-Based Risk (20-50% of spread)

**Method:** Temporal contact network
- Build graph where edges = vessel visits
- Edge weight = time between visits + vessel characteristics
- Filter: Only vessels that visited facilities in last 14 days

**Key factors:**
- Time between visits (shorter = higher risk)
- Vessel type (wellboat > service boat)
- Disinfection status (logged or not)
- Operation type (biomass transfer = highest risk)

In [ ]:
# Mock vessel visit data
vessel_visits = pd.DataFrame({
    'vessel_mmsi': ['257001000', '257001000', '257002000', '257003000', '257002000'],
    'vessel_name': ['LABRIDAE', 'LABRIDAE', 'OCEAN STAR', 'MARINE SERVICE', 'OCEAN STAR'],
    'vessel_type': ['wellboat', 'wellboat', 'wellboat', 'service', 'wellboat'],
    'facility_id': ['F001', 'F002', 'F001', 'F003', 'F004'],
    'visit_time': [
        datetime.now() - timedelta(days=5),
        datetime.now() - timedelta(days=3),
        datetime.now() - timedelta(days=2),
        datetime.now() - timedelta(days=1),
        datetime.now() - timedelta(hours=12)
    ],
    'operation': ['biomass_transfer', 'biomass_transfer', 'inspection', 'maintenance', 'biomass_transfer'],
    'disinfection_logged': [False, False, False, True, False]
})

def build_vessel_risk_graph(vessel_visits, facilities, max_days=14):
    """
    Build temporal graph showing vessel-mediated connections between facilities
    """
    G = nx.DiGraph()
    
    # Filter recent visits
    cutoff_time = datetime.now() - timedelta(days=max_days)
    recent_visits = vessel_visits[vessel_visits['visit_time'] > cutoff_time]
    
    # Sort by vessel and time
    recent_visits = recent_visits.sort_values(['vessel_mmsi', 'visit_time'])
    
    # Build edges (visit A -> visit B by same vessel)
    for vessel_id in recent_visits['vessel_mmsi'].unique():
        vessel_path = recent_visits[recent_visits['vessel_mmsi'] == vessel_id]
        
        for i in range(len(vessel_path) - 1):
            visit_from = vessel_path.iloc[i]
            visit_to = vessel_path.iloc[i + 1]
            
            time_delta_hours = (visit_to['visit_time'] - visit_from['visit_time']).total_seconds() / 3600
            
            # Risk factors
            time_risk = max(0, 100 - (time_delta_hours / 168) * 100)  # 7 days = 0 risk
            vessel_risk = 100 if visit_from['vessel_type'] == 'wellboat' else 50
            operation_risk = 100 if visit_from['operation'] == 'biomass_transfer' else 30
            disinfection_penalty = 0 if visit_from['disinfection_logged'] else 50
            
            # Check if source was infected
            source_infected = facilities[facilities['id'] == visit_from['facility_id']]['infected'].values[0]
            
            if source_infected:
                risk_score = min(100, (
                    0.3 * time_risk +
                    0.2 * vessel_risk +
                    0.3 * operation_risk +
                    0.2 * disinfection_penalty
                ))
                
                G.add_edge(
                    visit_from['facility_id'],
                    visit_to['facility_id'],
                    vessel=vessel_id,
                    vessel_name=visit_from['vessel_name'],
                    risk_score=risk_score,
                    time_delta_hours=time_delta_hours,
                    disinfected=visit_from['disinfection_logged']
                )
    
    return G

vessel_graph = build_vessel_risk_graph(vessel_visits, facilities)

print(f"🚢 Vessel Risk Network:")
print(f"   Nodes (facilities): {vessel_graph.number_of_nodes()}")
print(f"   Edges (vessel connections): {vessel_graph.number_of_edges()}")
print(f"\n   High-risk connections:")
for source, target, data in vessel_graph.edges(data=True):
    if data['risk_score'] > 50:
        source_name = facilities[facilities['id'] == source]['name'].values[0]
        target_name = facilities[facilities['id'] == target]['name'].values[0]
        print(f"   ⚠️  {source_name} → {target_name}")
        print(f"      Vessel: {data['vessel_name']}")
        print(f"      Risk: {data['risk_score']:.1f}/100")
        print(f"      Time between visits: {data['time_delta_hours']:.1f}h")
        print(f"      Disinfected: {'✅' if data['disinfected'] else '❌'}")

## 4. Biological Susceptibility

Even with exposure (current or vessel), infection depends on:
- **Temperature**: Cold water = higher ILA risk
- **Biomass**: Overcrowded = higher stress = higher risk
- **Production phase**: Smolt more susceptible than adult
- **Historical infections**: Recent infection = partial immunity OR weakened fish

In [ ]:
def calculate_biological_susceptibility(facility, disease_type='ILA'):
    """
    Calculate how likely a facility is to develop disease given exposure
    
    Returns: Susceptibility multiplier (0.0 - 2.0)
    """
    susceptibility = 1.0
    
    # Temperature factor (ILA thrives in cold water 4-8°C)
    if disease_type == 'ILA':
        temp = facility['water_temp_c']
        if 4 <= temp <= 8:
            temp_factor = 1.5  # Optimal for ILA
        elif temp < 4 or temp > 12:
            temp_factor = 0.5  # Low risk
        else:
            temp_factor = 1.0
        susceptibility *= temp_factor
    
    # Biomass stress factor (higher biomass = higher risk)
    biomass = facility['biomass_tons']
    if biomass > 900:
        biomass_factor = 1.3  # Overcrowded
    elif biomass < 600:
        biomass_factor = 0.8  # Low density
    else:
        biomass_factor = 1.0
    susceptibility *= biomass_factor
    
    # Production phase (mock - in reality from API)
    # Assuming all are in grow-out phase (multiplier = 1.0)
    # Smolt phase would be 1.5, adult harvest phase would be 0.7
    
    return round(susceptibility, 2)

# Calculate for each at-risk facility
print("🧬 Biological Susceptibility Analysis:\n")
for _, facility in at_risk_facilities.iterrows():
    susceptibility = calculate_biological_susceptibility(facility)
    print(f"   {facility['name']}:")
    print(f"      Temperature: {facility['water_temp_c']}°C")
    print(f"      Biomass: {facility['biomass_tons']} tons")
    print(f"      Susceptibility: {susceptibility}x (1.0 = baseline)")
    print()

## 5. Combined Risk Score (Hierarchical Model)

**Formula:**
```
Final Risk = max(
    Current_Exposure * Biological_Susceptibility,
    Vessel_Exposure * Biological_Susceptibility
) * Confidence_Factor
```

We use `max()` because spread happens via **either** route (not additive).

In [ ]:
def calculate_combined_risk_score(facility, infected_facilities, vessel_graph, current_data):
    """
    Calculate final outbreak probability for a facility
    
    Returns: Dict with risk score, breakdown, and confidence
    """
    # Get biological susceptibility
    susceptibility = calculate_biological_susceptibility(facility)
    
    # Calculate current-based risk (max from all infected sources)
    current_risk = 0
    for _, infected in infected_facilities.iterrows():
        exposure = calculate_current_exposure(
            infected, 
            facility,
            current_velocity_ms=current_data.get('velocity', 0.15),
            current_direction_deg=current_data.get('direction', 90)
        )
        current_risk = max(current_risk, exposure)
    
    # Calculate vessel-based risk
    vessel_risk = 0
    if facility['id'] in vessel_graph.nodes():
        # Get all incoming edges from infected facilities
        for source, target, data in vessel_graph.in_edges(facility['id'], data=True):
            if facilities[facilities['id'] == source]['infected'].values[0]:
                vessel_risk = max(vessel_risk, data['risk_score'])
    
    # Apply biological multiplier
    current_risk_adjusted = current_risk * susceptibility
    vessel_risk_adjusted = vessel_risk * susceptibility
    
    # Take maximum (not additive - spread happens via one route)
    final_risk = max(current_risk_adjusted, vessel_risk_adjusted)
    
    # Determine dominant pathway
    if current_risk_adjusted > vessel_risk_adjusted:
        dominant_pathway = 'ocean_current'
        pathway_confidence = 0.8 if current_risk > 50 else 0.6
    elif vessel_risk_adjusted > 0:
        dominant_pathway = 'vessel_transfer'
        pathway_confidence = 0.9  # Vessel data is more certain
    else:
        dominant_pathway = 'none'
        pathway_confidence = 0
    
    return {
        'facility_id': facility['id'],
        'facility_name': facility['name'],
        'final_risk_score': round(min(100, final_risk), 2),
        'risk_level': 'HIGH' if final_risk > 60 else 'MODERATE' if final_risk > 30 else 'LOW',
        'breakdown': {
            'current_exposure': round(current_risk, 2),
            'vessel_exposure': round(vessel_risk, 2),
            'biological_susceptibility': susceptibility,
            'current_risk_adjusted': round(current_risk_adjusted, 2),
            'vessel_risk_adjusted': round(vessel_risk_adjusted, 2)
        },
        'dominant_pathway': dominant_pathway,
        'confidence': pathway_confidence
    }

# Calculate for all at-risk facilities
infected_facilities = facilities[facilities['infected']]
current_data = {'velocity': 0.25, 'direction': 85}  # From NorKyst-800

risk_results = []
for _, facility in at_risk_facilities.iterrows():
    result = calculate_combined_risk_score(facility, infected_facilities, vessel_graph, current_data)
    risk_results.append(result)

# Sort by risk score
risk_results.sort(key=lambda x: x['final_risk_score'], reverse=True)

print("=" * 70)
print("📊 OUTBREAK PROBABILITY FORECAST - Next 7 Days")
print("=" * 70)
print()

for result in risk_results:
    emoji = '🔴' if result['risk_level'] == 'HIGH' else '🟡' if result['risk_level'] == 'MODERATE' else '🟢'
    print(f"{emoji} {result['facility_name']}")
    print(f"   Overall Risk: {result['final_risk_score']}/100 ({result['risk_level']})")
    print(f"   Confidence: {result['confidence']*100:.0f}%")
    print(f"   Primary Threat: {result['dominant_pathway'].replace('_', ' ').title()}")
    print(f"   Breakdown:")
    print(f"      • Ocean Current Exposure: {result['breakdown']['current_exposure']:.1f}")
    print(f"      • Vessel Exposure: {result['breakdown']['vessel_exposure']:.1f}")
    print(f"      • Biological Susceptibility: {result['breakdown']['biological_susceptibility']}x")
    print()

## 6. Attribution Analysis (Explaining Outbreaks)

**When a new outbreak is detected, answer: WHY?**

Method: Counterfactual analysis
- Calculate risk WITH all factors
- Calculate risk WITHOUT ocean currents
- Calculate risk WITHOUT vessel movements
- The factor with biggest delta is the probable cause

In [ ]:
def attribute_outbreak_source(outbreak_facility, infected_facilities, vessel_graph, current_data):
    """
    Explain what most likely caused an outbreak using counterfactual analysis
    """
    # Full risk calculation
    full_risk = calculate_combined_risk_score(outbreak_facility, infected_facilities, vessel_graph, current_data)
    
    # Counterfactual 1: No ocean currents (set velocity to 0)
    no_current_data = {'velocity': 0, 'direction': 0}
    no_current_risk = calculate_combined_risk_score(outbreak_facility, infected_facilities, vessel_graph, no_current_data)
    
    # Counterfactual 2: No vessel movements (empty graph)
    empty_graph = nx.DiGraph()
    no_vessel_risk = calculate_combined_risk_score(outbreak_facility, infected_facilities, empty_graph, current_data)
    
    # Calculate contribution percentages
    current_contribution = full_risk['breakdown']['current_risk_adjusted']
    vessel_contribution = full_risk['breakdown']['vessel_risk_adjusted']
    total_contribution = current_contribution + vessel_contribution
    
    if total_contribution > 0:
        current_pct = (current_contribution / total_contribution) * 100
        vessel_pct = (vessel_contribution / total_contribution) * 100
    else:
        current_pct = vessel_pct = 50.0
    
    # Find specific sources
    current_sources = []
    for _, inf in infected_facilities.iterrows():
        exposure = calculate_current_exposure(inf, outbreak_facility, 
                                              current_data['velocity'], current_data['direction'])
        if exposure > 30:
            current_sources.append({
                'facility': inf['name'],
                'disease': inf['disease'],
                'exposure': exposure
            })
    
    vessel_sources = []
    if outbreak_facility['id'] in vessel_graph.nodes():
        for source, _, data in vessel_graph.in_edges(outbreak_facility['id'], data=True):
            if facilities[facilities['id'] == source]['infected'].values[0]:
                source_name = facilities[facilities['id'] == source]['name'].values[0]
                vessel_sources.append({
                    'facility': source_name,
                    'vessel': data['vessel_name'],
                    'risk': data['risk_score'],
                    'time_delta_hours': data['time_delta_hours'],
                    'disinfected': data['disinfected']
                })
    
    return {
        'outbreak_facility': outbreak_facility['name'],
        'attribution': {
            'ocean_current_pct': round(current_pct, 1),
            'vessel_transfer_pct': round(vessel_pct, 1),
            'other_pct': round(100 - current_pct - vessel_pct, 1)
        },
        'current_sources': current_sources,
        'vessel_sources': vessel_sources,
        'recommendation': 'Prioritize vessel biosecurity' if vessel_pct > 60 else 'Monitor ocean currents' if current_pct > 60 else 'Multiple risk factors'
    }

# Simulate outbreak at Nordkapp
simulated_outbreak = facilities[facilities['name'] == 'Nordkapp'].iloc[0]
attribution = attribute_outbreak_source(simulated_outbreak, infected_facilities, vessel_graph, current_data)

print("="*70)
print(f"🔬 OUTBREAK ATTRIBUTION ANALYSIS - {attribution['outbreak_facility']}")
print("="*70)
print()
print("PROBABLE SOURCES:")
print()
print(f"1. Ocean Currents: {attribution['attribution']['ocean_current_pct']}%")
for source in attribution['current_sources']:
    print(f"   └─ From: {source['facility']} ({source['disease']})")
    print(f"      Exposure: {source['exposure']:.1f}/100")
print()
print(f"2. Vessel Transfer: {attribution['attribution']['vessel_transfer_pct']}%")
for source in attribution['vessel_sources']:
    print(f"   └─ Vessel: {source['vessel']}")
    print(f"      From: {source['facility']}")
    print(f"      Time between visits: {source['time_delta_hours']:.1f}h")
    print(f"      Disinfection: {'✅' if source['disinfected'] else '❌ NOT LOGGED'}")
    print(f"      Risk: {source['risk']:.1f}/100")
print()
print(f"3. Other factors: {attribution['attribution']['other_pct']}%")
print("   (Wild fish, equipment, unknown)")
print()
print(f"📋 RECOMMENDATION: {attribution['recommendation']}")

## 7. Operational Application

### A. Smart Route Validation

In [ ]:
def validate_route(planned_route, facilities):
    """
    Validate a planned vessel route and issue warnings
    
    Args:
        planned_route: List of facility IDs in visit order
        facilities: DataFrame with facility data
    
    Returns:
        Dict with warnings and recommendations
    """
    warnings = []
    recommendations = []
    total_risk_score = 0
    
    for i in range(len(planned_route) - 1):
        from_id = planned_route[i]
        to_id = planned_route[i + 1]
        
        from_facility = facilities[facilities['id'] == from_id].iloc[0]
        to_facility = facilities[facilities['id'] == to_id].iloc[0]
        
        if from_facility['infected'] and not to_facility['infected']:
            risk = 90  # High risk
            warnings.append({
                'type': 'CRITICAL',
                'message': f"Visiting INFECTED facility {from_facility['name']} before HEALTHY facility {to_facility['name']}",
                'from': from_facility['name'],
                'to': to_facility['name'],
                'disease': from_facility['disease'],
                'risk': risk
            })
            recommendations.append(f"⚠️  REQUIRED: 48h quarantine + disinfection after {from_facility['name']}")
            total_risk_score += risk
        
        elif to_facility['infected']:
            # Visiting infected is OK if nothing follows OR if quarantine planned
            if i < len(planned_route) - 2:  # More visits after this
                next_facility = facilities[facilities['id'] == planned_route[i+2]].iloc[0]
                if not next_facility['infected']:
                    warnings.append({
                        'type': 'WARNING',
                        'message': f"Infected facility {to_facility['name']} not scheduled as last stop",
                        'recommendation': 'Move infected facilities to end of route'
                    })
                    recommendations.append(f"💡 Consider moving {to_facility['name']} to end of route")
    
    return {
        'warnings': warnings,
        'recommendations': recommendations,
        'total_risk_score': total_risk_score,
        'route_safe': len(warnings) == 0
    }

# Test route validation
planned_route = ['F003', 'F001', 'F002']  # Alta Vest → Hammerfest Nord (INFECTED) → Nordkapp

validation_result = validate_route(planned_route, facilities)

print("🚢 ROUTE VALIDATION")
print("=" * 50)
print(f"Route: {' → '.join([facilities[facilities['id'] == fid]['name'].values[0] for fid in planned_route])}")
print()
if validation_result['route_safe']:
    print("✅ Route is SAFE - No biosecurity violations")
else:
    print(f"⚠️  BIOSECURITY RISK DETECTED (Score: {validation_result['total_risk_score']}/100)")
    print()
    for warning in validation_result['warnings']:
        print(f"   🚨 {warning['type']}: {warning['message']}")
        if 'disease' in warning:
            print(f"      Disease: {warning['disease']}")
    print()
    print("RECOMMENDATIONS:")
    for rec in validation_result['recommendations']:
        print(f"   {rec}")

## 8. Production Architecture

### Database Schema

```sql
-- TimescaleDB for time-series data
CREATE TABLE ocean_currents (
    time TIMESTAMPTZ NOT NULL,
    lat FLOAT,
    lon FLOAT,
    u_velocity FLOAT,  -- eastward m/s
    v_velocity FLOAT,  -- northward m/s
    source VARCHAR(50)  -- 'norkyst800'
);
SELECT create_hypertable('ocean_currents', 'time');

-- PostGIS for spatial facility data
CREATE TABLE facilities (
    id VARCHAR(20) PRIMARY KEY,
    name VARCHAR(255),
    location GEOGRAPHY(POINT, 4326),
    infected BOOLEAN DEFAULT FALSE,
    disease VARCHAR(50),
    biomass_tons FLOAT,
    water_temp_c FLOAT,
    last_updated TIMESTAMPTZ
);
CREATE INDEX ON facilities USING GIST(location);

-- AIS vessel tracking
CREATE TABLE vessel_visits (
    id SERIAL PRIMARY KEY,
    vessel_mmsi VARCHAR(20),
    facility_id VARCHAR(20) REFERENCES facilities(id),
    visit_time TIMESTAMPTZ,
    operation VARCHAR(50),
    disinfection_logged BOOLEAN DEFAULT FALSE
);
CREATE INDEX ON vessel_visits(visit_time DESC);
CREATE INDEX ON vessel_visits(vessel_mmsi, visit_time);

-- Calculated risk scores (cached)
CREATE TABLE risk_assessments (
    facility_id VARCHAR(20) REFERENCES facilities(id),
    calculated_at TIMESTAMPTZ NOT NULL,
    risk_score FLOAT,
    risk_level VARCHAR(20),
    current_exposure FLOAT,
    vessel_exposure FLOAT,
    dominant_pathway VARCHAR(50),
    confidence FLOAT
);
SELECT create_hypertable('risk_assessments', 'calculated_at');
```

### API Endpoints (FastAPI)

```python
@app.get("/api/risk/forecast/{facility_id}")
async def get_risk_forecast(facility_id: str, days_ahead: int = 7):
    """Get outbreak probability forecast for a facility"""
    
@app.get("/api/risk/map")
async def get_risk_map(min_risk: float = 30):
    """Get all facilities above risk threshold for map visualization"""
    
@app.post("/api/risk/attribution")
async def analyze_outbreak(outbreak_data: OutbreakEvent):
    """Explain probable source of an outbreak"""
    
@app.post("/api/vessel/validate-route")
async def validate_vessel_route(route: List[str]):
    """Validate planned route and return warnings"""
    
@app.get("/api/alerts/daily")
async def get_daily_alerts():
    """Get facilities that crossed risk threshold in last 24h"""
```

### Scheduled Jobs

```python
# Daily 06:00: Recalculate all at-risk facilities
@scheduler.scheduled_job('cron', hour=6)
def daily_risk_calculation():
    infected = get_infected_facilities()
    at_risk = filter_at_risk_facilities(infected, radius_km=150)
    
    for facility in at_risk:
        risk = calculate_combined_risk_score(facility)
        save_risk_assessment(risk)
        
        if risk['final_risk_score'] > 70:
            send_alert_email(facility, risk)

# Hourly: Update ocean current data from NorKyst-800
@scheduler.scheduled_job('interval', hours=1)
def update_ocean_currents():
    data = fetch_norkyst800_latest()
    store_current_data(data)

# Every 15 min: Check vessel AIS positions
@scheduler.scheduled_job('interval', minutes=15)
def track_vessels():
    vessels = get_vessels_near_facilities(radius_km=1)
    for vessel in vessels:
        log_facility_visit(vessel)
```

## 9. Next Steps & Recommendations

### Phase 1 (Week 1-2): Foundation
✅ **Done:**
- NorKyst-800 integration
- Basic facility data
- API structure

🔨 **To Do:**
1. Add temperature data (Frost API)
2. Implement smart facility filtering
3. Build current exposure calculation endpoint
4. Create risk visualization on admin dashboard

### Phase 2 (Week 3-4): Operational
1. Vessel proximity detection from AIS
2. Route validation in vessel dashboard
3. Daily risk calculation job
4. Email alerts for high-risk facilities

### Phase 3 (Month 2): Advanced
1. Attribution analysis for outbreaks
2. Historical "what caused this?" tool
3. Confidence intervals on predictions
4. ML model training (once we have 1 month data)

### Key Design Principles
1. ⚡ **Performance:** Filter before calculating (don't process all 2689 facilities)
2. 🎯 **Accuracy:** Separate pathways (current vs vessel) for explainability
3. 📊 **Trust:** Show confidence levels, not just scores
4. 🔄 **Actionable:** Every prediction must suggest an action
5. 🧪 **Testable:** Track predictions vs actual outbreaks to improve model

### Critical Success Factors
- **Data Quality:** NorKyst-800 is good, but need reliable vessel visit logs
- **User Adoption:** Must be simple enough that operators use it daily
- **Validation:** Track prediction accuracy from Week 1
- **Feedback Loop:** Let users report "false alarms" to improve model